# Introducing Naive, Advanced, and Modular RAG with Oracle

**Copyright 2026, Denis Rothman**

This notebook introduces Naïve, Advanced, and Modular RAG. It marks a revolutionary conceptual shift in Generative AI architecture: **MAS-RAG (Multi-Agent Systems for Retrieval Augmented Generation)**.

### The Revolutionary Conceptual Shift: Bringing AI to the Data

Traditionally, RAG architectures required moving corporate data *out* of secure environments into external vector stores and processing engines. We are reversing this paradigm.

| Feature | **Traditional RAG (The Old Way)** | **Oracle MAS-RAG (The New Way)** |
| :--- | :--- | :--- |
| **Data Movement** | Moves data **to** the AI (External Vector DBs) | Moves AI **to** the Data (Oracle Database 23ai) |
| **Security** | High risk of leakage; data leaves trust boundary | **Zero Trust:** Data never leaves the database |
| **Latency** | Network latency between Vector Store & App | **Real-time:** Vectors & Data coexist locally |
| **Data Freshness** | Stale data (requires frequent re-indexing/ETL) | **Instant:** Updates are immediately retrievable |
| **Architecture** | Fragmented (App + Vector DB + SQL DB) | **Converged:** Single engine for SQL, JSON, & Vectors |

By implementing RAG directly within **Oracle**, we utilize **Oracle Database 23ai** capabilities. This ensures that the Multi-Agent Systems operate strictly within the corporate trust boundary, leveraging data where it lives securely.

---

### **Summary**

**Part 1: Foundations and Basic Implementation**
1.  **Environment:** Setup for secure integration.
2.  **Generator:** Utilizing LLMs with corporate context.
3.  **The Data:** A simulated **Oracle Database** environment (simulated via `db_records`) illustrating Integrated Vector logic.
4.  **Query:** Asking strategic questions about data sovereignty and architecture.

**Part 2: Advanced Techniques and Evaluation**
1.  Retrieval metrics
2.  Naive RAG
3.  Advanced RAG within Oracle
4.  Modular RAG Retriever

# Part 1: Foundations and Basic Implementation

# 1.The Environment

In [1]:
!pip install openai==2.14.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 12.8 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.24.0
    Uninstalling openai-2.24.0:
      Successfully uninstalled openai-2.24.0


## Retrieving API Key (Google Secrets)

In [2]:
# Cell 2: Imports and API Key Setup
# We will use the OpenAI library to interact with the LLM and Google Colab's
# secret manager to securely access your API key.

import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    api_key = userdata.get("API_KEY")
    if not api_key:
        raise userdata.SecretNotFoundError("API_KEY not found.")

    # Set environment variable for downstream tools/libraries
    os.environ["OPENAI_API_KEY"] = api_key

    # Create client (will read from OPENAI_API_KEY)
    client = OpenAI()
    print("OpenAI API key loaded and environment variable set successfully.")

except userdata.SecretNotFoundError:
    print('Secret "API_KEY" not found.')
    print('Please add your OpenAI API key to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the API key: {e}")


OpenAI API key loaded and environment variable set successfully.


# 2.The Generator


In [3]:
import openai
from openai import OpenAI

client = OpenAI()
gptmodel="gpt-5.2"

def call_llm_with_full_text(itext):
    # Join all lines to form a single string
    text_input = '\n'.join(itext)
    prompt = f"Please elaborate on the following content:\n{text_input}"

    try:
      response = client.chat.completions.create(
         model=gptmodel,
         messages=[
            {"role": "system", "content": "You are an expert Natural Language Processing exercise expert."},
            {"role": "assistant", "content": "1.You can explain read the input and answer in detail"},
            {"role": "user", "content": prompt}
         ],
         temperature=0.1  # Add the temperature parameter here and other parameters you need
        )
      return response.choices[0].message.content.strip()
    except Exception as e:
        return str(e)

## Formatted response

In [4]:
import textwrap

def print_formatted_response(response):
    # Define the width for wrapping the text
    wrapper = textwrap.TextWrapper(width=80)  # Set to 80 columns wide, but adjust as needed
    wrapped_text = wrapper.fill(text=response)

    # Print the formatted response with a header and footer
    print("Response:")
    print("---------------")
    print(wrapped_text)
    print("---------------\n")

 # 3.The Data simulating an Oracle db retrieval

In [5]:
db_records = [
    "Retrieval Augmented Generation (RAG) is a hybrid AI approach that combines neural language models with secure retrieval systems.",
    "The Oracle strategy for RAG represents a paradigm shift: instead of moving data to external vector stores, it brings AI vector search directly to the database.",
    "This methodology ensures that sensitive corporate data never leaves the secure governance boundary of the Oracle environment.",
    "At the core of this architecture is the Oracle Database 23ai, which integrates vector search natively alongside relational data.",
    "Unlike external platforms that require data replication (ETL), Oracle allows for 'Integrated Vectorization', meaning vectors and business data coexist in real-time.",
    "The retrieval system utilizes Oracle AI Vector Search to fetch contextually relevant documents without the latency of external API calls.",
    "When a query is received, the Oracle system performs a similarity search directly on the resident data, ensuring zero data leakage.",
    "This component merges the retrieval outputs with the generative model, ensuring the final response is grounded in the organization's single source of truth.",
    "By keeping data in Oracle, the system eliminates the complexity of synchronizing a separate vector database with the operational database.",
    "The integrator seamlessly incorporates the retrieved Oracle data into the final text output, enhancing accuracy and compliance.",
    "This approach is particularly critical for industries like finance, healthcare, and government where data sovereignty is non-negotiable.",
    "Oracle AI Vector Search supports various distance metrics and indexing methods tailored for high-performance corporate workloads.",
    "The Multi-Agent System (MAS) can now orchestrate complex tasks by querying the Oracle database directly for both structured (SQL) and unstructured (Vector) data.",
    "This response is influenced by real-time corporate data, making it far more accurate than models relying on stale, externally indexed data.",
    "Retrieval Augmented Generation (RAG) in Oracle adapts dynamically; as soon as a record is updated in the database, it is immediately available for retrieval.",
    "This eliminates the 'stale data' problem common in architectures that rely on external vector stores like Pinecone or FAISS.",
    "With access to the vast, secure repository of Oracle data, the RAG system provides nuanced answers grounded in the company's actual operational reality.",
    "The challenge of maintaining a separate, high-quality database of retrievable texts is solved by converging vectors and data into one Oracle platform.",
    "Furthermore, Oracle's robust security features (Row-Level Security, auditing) automatically apply to the vector retrieval process.",
    "In summary, Oracle-based RAG represents the maturity of AI: merging the best of generative technologies with the security and reliability of enterprise data systems.",
    "An Oracle Vector Store is not a separate silo; it is a capability of the converged database that treats vectors as a native data type."
]

In [6]:
import textwrap
paragraph = ' '.join(db_records)
wrapped_text = textwrap.fill(paragraph, width=80)
print(wrapped_text)

Retrieval Augmented Generation (RAG) is a hybrid AI approach that combines
neural language models with secure retrieval systems. The Oracle strategy for
RAG represents a paradigm shift: instead of moving data to external vector
stores, it brings AI vector search directly to the database. This methodology
ensures that sensitive corporate data never leaves the secure governance
boundary of the Oracle environment. At the core of this architecture is the
Oracle Database 23ai, which integrates vector search natively alongside
relational data. Unlike external platforms that require data replication (ETL),
Oracle allows for 'Integrated Vectorization', meaning vectors and business data
coexist in real-time. The retrieval system utilizes Oracle AI Vector Search to
fetch contextually relevant documents without the latency of external API calls.
When a query is received, the Oracle system performs a similarity search
directly on the resident data, ensuring zero data leakage. This component merges

# 4.The Query

In [7]:
# A query designed to trigger the retrieval of "Integrated Vectorization" and "Data Sovereignty" concepts
query = "How does the Oracle RAG approach differ from traditional vector stores regarding data security?"

Generation without augmentation

In [8]:
# Call the function and print the result
llm_response = call_llm_with_full_text(query)
print_formatted_response(llm_response)

Response:
---------------
Oracle’s RAG (Retrieval-Augmented Generation) approach differs from many
“traditional” vector-store setups mainly in **where the data lives, how access
is controlled, and how governance is enforced**. Here are the key security-
related differences.  ## 1) Data stays inside the database security boundary
**Traditional vector store pattern:**   Often involves exporting documents (or
chunks) from an enterprise system into a separate vector database/service. That
creates **another copy of sensitive data** (and embeddings) outside the primary
governed system.  **Oracle RAG approach (typical):**   Vectors/embeddings and
source text can be stored and queried **inside Oracle Database** (e.g., using
Oracle’s vector capabilities). This keeps data within the **same security
perimeter** as the rest of your enterprise data—reducing data sprawl and the
risk introduced by additional systems.  ## 2) Uses mature database access
controls (RBAC, VPD/FGAC, row-level security) **T

# Part 2: Advanced Techniques and Evaluation

# 1.Retrieval Metrics

## Cosine Similarity

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(text1, text2):
    # Use the default settings to match the book's explanation
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform([text1, text2])
    similarity = cosine_similarity(tfidf[0:1], tfidf[1:2])
    return similarity[0][0]

## Enhanced Similarity

In [10]:
import spacy
import nltk
nltk.download('wordnet')
from nltk.corpus import wordnet
from collections import Counter
import numpy as np

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name())
    return synonyms

def preprocess_text(text):
    doc = nlp(text.lower())
    lemmatized_words = []
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        lemmatized_words.append(token.lemma_)
    return lemmatized_words

def expand_with_synonyms(words):
    expanded_words = words.copy()
    for word in words:
        expanded_words.extend(get_synonyms(word))
    return expanded_words

def calculate_enhanced_similarity(text1, text2):
    # Preprocess and tokenize texts
    words1 = preprocess_text(text1)
    words2 = preprocess_text(text2)

    # Expand with synonyms
    words1_expanded = expand_with_synonyms(words1)
    words2_expanded = expand_with_synonyms(words2)

    # Count word frequencies
    freq1 = Counter(words1_expanded)
    freq2 = Counter(words2_expanded)

    # Create a set of all unique words
    unique_words = set(freq1.keys()).union(set(freq2.keys()))

    # Create frequency vectors
    vector1 = [freq1[word] for word in unique_words]
    vector2 = [freq2[word] for word in unique_words]

    # Convert lists to numpy arrays
    vector1 = np.array(vector1)
    vector2 = np.array(vector2)

    # Calculate cosine similarity
    cosine_similarity = np.dot(vector1, vector2) / (np.linalg.norm(vector1) * np.linalg.norm(vector2))

    return cosine_similarity

[nltk_data] Downloading package wordnet to /root/nltk_data...


# 2.Naive RAG

## Keyword search and matching

In [11]:
def find_best_match_keyword_search(query, db_records):
    best_score = 0
    best_record = None

    # Split the query into individual keywords
    query_keywords = set(query.lower().split())

    # Iterate through each record in db_records
    for record in db_records:
        # Split the record into keywords
        record_keywords = set(record.lower().split())

        # Calculate the number of common keywords
        common_keywords = query_keywords.intersection(record_keywords)
        current_score = len(common_keywords)

        # Update the best score and record if the current score is higher
        if current_score > best_score:
            best_score = current_score
            best_record = record

    return best_score, best_record

# Assuming 'query' and 'db_records' are defined in previous cells in your Colab notebook
best_keyword_score, best_matching_record = find_best_match_keyword_search(query, db_records)

print(f"Best Keyword Score: {best_keyword_score}")
print_formatted_response(best_matching_record)

Best Keyword Score: 5
Response:
---------------
The Oracle strategy for RAG represents a paradigm shift: instead of moving data
to external vector stores, it brings AI vector search directly to the database.
---------------



## Metrics

In [12]:
# Cosine Similarity
score = calculate_cosine_similarity(query, best_matching_record)
print(f"Best Cosine Similarity Score: {score:.3f}")

Best Cosine Similarity Score: 0.243


In [13]:
# Enhanced Similarity
response = best_matching_record
print(query,": ", response)
similarity_score = calculate_enhanced_similarity(query, response)
print(f"Enhanced Similarity:, {similarity_score:.3f}")

How does the Oracle RAG approach differ from traditional vector stores regarding data security? :  The Oracle strategy for RAG represents a paradigm shift: instead of moving data to external vector stores, it brings AI vector search directly to the database.
Enhanced Similarity:, 0.503


## Augmented input

In [14]:
augmented_input=query+ ": "+ best_matching_record

In [15]:
print_formatted_response(augmented_input)

Response:
---------------
How does the Oracle RAG approach differ from traditional vector stores regarding
data security?: The Oracle strategy for RAG represents a paradigm shift: instead
of moving data to external vector stores, it brings AI vector search directly to
the database.
---------------



## Generation

In [16]:
# Call the function and print the result
llm_response = call_llm_with_full_text(augmented_input)
print_formatted_response(llm_response)

Response:
---------------
Oracle’s RAG (Retrieval-Augmented Generation) approach differs from many
traditional vector-store setups mainly in **where the vectors live and where
retrieval happens**, and that has direct consequences for **data security and
governance**.  ## Traditional vector-store pattern (common in many RAG stacks)
In a typical architecture:  1. **Data is exported from the primary system of
record** (often an enterprise database or document repository). 2. It is
**chunked and embedded**. 3. The embeddings (and often some text/metadata) are
**loaded into an external vector database/service**. 4. Applications query that
external store for retrieval.  ### Security implications - **Data duplication
and extra attack surface**: You now have another system holding sensitive
representations of your data (embeddings + metadata, sometimes raw text). -
**More places to secure**: IAM, encryption, network controls, backups, logging,
patching, incident response—across multiple platfo

# 3.Advanced RAG

## 3.1.Vector search

### Search function

In [17]:
def find_best_match(text_input, records):
    best_score = 0
    best_record = None
    for record in records:
        current_score = calculate_cosine_similarity(text_input, record)
        if current_score > best_score:
            best_score = current_score
            best_record = record
    return best_score, best_record

In [18]:
best_similarity_score, best_matching_record = find_best_match(query, db_records)

In [19]:
print_formatted_response(best_matching_record)

Response:
---------------
The Oracle strategy for RAG represents a paradigm shift: instead of moving data
to external vector stores, it brings AI vector search directly to the database.
---------------



### Metrics

In [20]:
print(f"Best Cosine Similarity Score: {best_similarity_score:.3f}")

Best Cosine Similarity Score: 0.243


In [21]:
# Enhanced Similarity
response = best_matching_record
print(query,": ", response)
similarity_score = calculate_enhanced_similarity(query, best_matching_record)
print(f"Enhanced Similarity:, {similarity_score:.3f}")

How does the Oracle RAG approach differ from traditional vector stores regarding data security? :  The Oracle strategy for RAG represents a paradigm shift: instead of moving data to external vector stores, it brings AI vector search directly to the database.
Enhanced Similarity:, 0.503


### Augmented input

In [22]:
augmented_input=query+": "+best_matching_record

In [23]:
print_formatted_response(augmented_input)

Response:
---------------
How does the Oracle RAG approach differ from traditional vector stores regarding
data security?: The Oracle strategy for RAG represents a paradigm shift: instead
of moving data to external vector stores, it brings AI vector search directly to
the database.
---------------



### Generation

In [24]:
# Call the function and print the result
llm_response = call_llm_with_full_text(augmented_input)
print_formatted_response(llm_response)

Response:
---------------
Oracle’s RAG (Retrieval-Augmented Generation) approach differs from many
traditional “external vector store” setups mainly in **where the vectors live
and where retrieval happens**, which has direct consequences for **data security
and governance**.  ## 1) “Bring AI to the data” vs. “Move data to the AI”
**Traditional vector-store pattern** - You typically **extract documents** (or
chunks) from your core systems (often a database), - **generate embeddings**,
and - **push the text + embeddings** into a separate vector database/service.
This creates an additional copy of potentially sensitive content outside the
primary system of record.  **Oracle RAG strategy** - Oracle’s approach
emphasizes **running vector search inside the database** (e.g., Oracle Database
with vector capabilities), - so you **don’t need to export data** to an external
vector store to enable semantic retrieval.  **Security implication:** fewer data
copies and fewer places where sensitive dat

## 3.2.Index-based search

### Search Function

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def setup_vectorizer(records):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(records)
    return vectorizer, tfidf_matrix

def find_best_match_indexed(query, vectorizer, tfidf_matrix):
    query_tfidf = vectorizer.transform([query])
    similarities = cosine_similarity(query_tfidf, tfidf_matrix)
    best_index = similarities.argmax()  # Get the index of the highest similarity score
    best_score = similarities[0, best_index]
    return best_score, best_index

vectorizer, tfidf_matrix = setup_vectorizer(db_records)

best_similarity_score, best_index = find_best_match_indexed(query, vectorizer, tfidf_matrix)
best_matching_record = db_records[best_index]

print_formatted_response(best_matching_record)

Response:
---------------
The Oracle strategy for RAG represents a paradigm shift: instead of moving data
to external vector stores, it brings AI vector search directly to the database.
---------------



### Metrics

In [26]:
# Cosine Similarity
print(f"Best Cosine Similarity Score: {best_similarity_score:.3f}")
print_formatted_response(best_matching_record)

Best Cosine Similarity Score: 0.318
Response:
---------------
The Oracle strategy for RAG represents a paradigm shift: instead of moving data
to external vector stores, it brings AI vector search directly to the database.
---------------



In [27]:
# Enhanced Similarity
response = best_matching_record
print(query,": ", response)
similarity_score = calculate_enhanced_similarity(query, response)
print(f"Enhanced Similarity:, {similarity_score:.3f}")

How does the Oracle RAG approach differ from traditional vector stores regarding data security? :  The Oracle strategy for RAG represents a paradigm shift: instead of moving data to external vector stores, it brings AI vector search directly to the database.
Enhanced Similarity:, 0.503


Feature extraction

In [28]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

def setup_vectorizer(records):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(records)

    # Convert the TF-IDF matrix to a DataFrame for display purposes
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

    # Display the DataFrame
    print(tfidf_df)

    return vectorizer, tfidf_matrix

vectorizer, tfidf_matrix = setup_vectorizer(db_records)

        23ai    access  accuracy  accurate    actual   adapts     agent  \
0   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
1   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
2   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
3   0.290558  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
4   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
5   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
6   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
7   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
8   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
9   0.000000  0.000000  0.283723  0.000000  0.000000  0.00000  0.000000   
10  0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
11  0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
12  0.000000  0.000000  0

### Augmented input

In [29]:
augmented_input=query+": "+best_matching_record

In [30]:
print_formatted_response(augmented_input)

Response:
---------------
How does the Oracle RAG approach differ from traditional vector stores regarding
data security?: The Oracle strategy for RAG represents a paradigm shift: instead
of moving data to external vector stores, it brings AI vector search directly to
the database.
---------------



### Generation

In [31]:
# Call the function and print the result
llm_response = call_llm_with_full_text(augmented_input)
print_formatted_response(llm_response)

Response:
---------------
Oracle’s RAG (Retrieval-Augmented Generation) approach differs from many
traditional “external vector store” setups mainly in **where the vectors live
and where retrieval happens**—and that has direct consequences for **data
security, governance, and compliance**.  ## 1) “Bring AI to the data” vs. “Move
data to the AI” **Traditional vector store pattern** - You typically **extract
documents/data** from your core systems (often a database), - **chunk** them, -
create **embeddings**, and - **store those embeddings + metadata (and sometimes
the text itself)** in a separate vector database/service.  This means sensitive
information is **copied** into another system, creating additional security and
compliance surface area.  **Oracle RAG strategy** - Oracle’s framing is: **don’t
move data out to an external vector store**. - Instead, you run **vector search
inside the database** (where the data already resides).  **Security
implication:** fewer copies of sensitive 

# 4.Modular RAG

Modular RAG can combine methods. For example:

**keyword search**:Searches through each document to find the one that best matches the keyword(s).

**vector search**: Searches through each document and calculates similarity.

**indexed search**: Uses a precomputed index (TF-IDF matrix) to compute cosine similarities.

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class RetrievalComponent:
    def __init__(self, method='vector'):
        self.method = method
        if self.method == 'vector' or self.method == 'indexed':
            self.vectorizer = TfidfVectorizer()
            self.tfidf_matrix = None

    def fit(self, records):
      self.documents = records  # Initialize self.documents here
      if self.method == 'vector' or self.method == 'indexed':
        self.tfidf_matrix = self.vectorizer.fit_transform(records)

    def retrieve(self, query):
        if self.method == 'keyword':
            return self.keyword_search(query)
        elif self.method == 'vector':
            return self.vector_search(query)
        elif self.method == 'indexed':
            return self.indexed_search(query)

    def keyword_search(self, query):
        best_score = 0
        best_record = None
        query_keywords = set(query.lower().split())
        for index, doc in enumerate(self.documents):
            doc_keywords = set(doc.lower().split())
            common_keywords = query_keywords.intersection(doc_keywords)
            score = len(common_keywords)
            if score > best_score:
                best_score = score
                best_record = self.documents[index]
        return best_record

    def vector_search(self, query):
        query_tfidf = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_tfidf, self.tfidf_matrix)
        best_index = similarities.argmax()
        return self.documents[best_index]

    def indexed_search(self, query):
        # Assuming the tfidf_matrix is precomputed and stored
        query_tfidf = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_tfidf, self.tfidf_matrix)
        best_index = similarities.argmax()
        return self.documents[best_index]

### Modular RAG Strategies

In [33]:
# Usage example
retrieval = RetrievalComponent(method='vector')  # Choose from 'keyword', 'vector', 'indexed'
retrieval.fit(db_records)
best_matching_record = retrieval.retrieve(query)

print_formatted_response(best_matching_record)

Response:
---------------
The Oracle strategy for RAG represents a paradigm shift: instead of moving data
to external vector stores, it brings AI vector search directly to the database.
---------------



### Metrics

In [34]:
# Cosine Similarity
print(f"Best Cosine Similarity Score: {best_similarity_score:.3f}")
print_formatted_response(best_matching_record)

Best Cosine Similarity Score: 0.318
Response:
---------------
The Oracle strategy for RAG represents a paradigm shift: instead of moving data
to external vector stores, it brings AI vector search directly to the database.
---------------



In [35]:
# Enhanced Similarity
response = best_matching_record
print(query,": ", response)
similarity_score = calculate_enhanced_similarity(query, response)
print("Enhanced Similarity:", similarity_score)

How does the Oracle RAG approach differ from traditional vector stores regarding data security? :  The Oracle strategy for RAG represents a paradigm shift: instead of moving data to external vector stores, it brings AI vector search directly to the database.
Enhanced Similarity: 0.5031579073491305


### Augmented Input

In [36]:
augmented_input=query+ " "+ best_matching_record

In [37]:
print_formatted_response(augmented_input)

Response:
---------------
How does the Oracle RAG approach differ from traditional vector stores regarding
data security? The Oracle strategy for RAG represents a paradigm shift: instead
of moving data to external vector stores, it brings AI vector search directly to
the database.
---------------



### Generation

In [38]:
# Call the function and print the result
llm_response = call_llm_with_full_text(augmented_input)
print_formatted_response(llm_response)

Response:
---------------
Oracle’s RAG approach differs from many traditional “external vector store”
setups mainly in **where the vectors live and how security is enforced**.  ##
Traditional vector-store pattern (common today) In many RAG architectures, you:
1. Extract documents from your primary system (often a database). 2. Generate
embeddings. 3. **Copy** the text/chunks + embeddings into a **separate vector
database/service**. 4. At query time, you retrieve from that external store and
then pass results to the LLM.  ### Security implications - **Data duplication**:
Sensitive content is replicated outside the system of record, increasing the
attack surface. - **New security boundary**: You must secure *another* platform
(network access, IAM, encryption, backups, logging, patching). - **Policy
drift**: The external store may not enforce the same fine-grained controls as
the source database (e.g., row-level security, column masking, tenant
isolation). - **More integration risk**: ETL